# MBA Capstone Summary — Generative AI Evaluation in Public Health (Malaria)

This notebook is a **portfolio summary** of my final MBA project (USP/Esalq, 2025).  
It highlights methodology, key findings and practical implications using the processed outputs generated by the original script:

- `code/TCC_MBA_DSA_USP_MALARIA.py`

> Note: The full research report was written in Portuguese and this notebook presents an English executive summary for portfolio review.


## 1. Project context

**Research question**

How close are responses from popular generative AI systems to official WHO (OMS) content on malaria, considering readability, lexical/semantic similarity and consistency across topics?

**Data design**

- 10 AI systems compared against OMS reference responses
- 12 questions across 4 topic groups
- Final analytical unit for topic-level modeling: `AI × topic`


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE = Path("..")
RESULTS = BASE / "results"
FIGURES = BASE / "figures"

df_agg_ranked = pd.read_excel(RESULTS / "df_agg_ranked.xlsx")
df_agg_topic = pd.read_excel(RESULTS / "df_agg_topic.xlsx")
df_agg_topic_cluster = pd.read_excel(RESULTS / "df_agg_topic_cluster.xlsx")
df_oms_metrics = pd.read_excel(RESULTS / "df_oms_metrics.xlsx")
df_all_responses = pd.read_excel(RESULTS / "df_all_responses.xlsx")

print("Loaded:")
print("- df_agg_ranked:", df_agg_ranked.shape)
print("- df_agg_topic:", df_agg_topic.shape)
print("- df_agg_topic_cluster:", df_agg_topic_cluster.shape)
print("- df_oms_metrics:", df_oms_metrics.shape)
print("- df_all_responses:", df_all_responses.shape)

ImportError: `Import openpyxl` failed.  Use pip or conda to install the openpyxl package.

In [ ]:
print("AI models in evaluation:", sorted(df_agg_ranked["AI"].unique()))
print("Topics:", sorted(df_agg_topic["topic"].unique()))
print("Questions per topic (OMS):")
display(df_oms_metrics.groupby("topic")["question"].count().rename("n_questions").reset_index())

## 2. Methodology (executive view)

The original study combines text analytics and multivariate statistics:

1. **Readability:** Flesch Reading Ease, Flesch-Kincaid Grade Level  
2. **Similarity:** Cosine (TF-IDF), Jaccard, Levenshtein distance  
3. **Composite ranking:** min-max normalization + aggregate score  
4. **Topic-level analysis:** ranking by topic (`AI × topic`)  
5. **Unsupervised learning:** K-Means (`k` selected using Elbow + Silhouette)  
6. **Validation and interpretation:** ANOVA, chi-square and MCA (2D/3D perceptual maps)


## 3. Overall AI ranking

The table below shows the final normalized score (`score_final`) used to rank systems globally across all metrics.


In [ ]:
ranking_cols = [
    "ranking",
    "AI",
    "score_final",
    "flesch_reading_ease",
    "cosine_similarity",
    "jaccard_similarity",
]

df_rank = df_agg_ranked.sort_values("ranking").copy()
display(df_rank[ranking_cols])

top3 = df_rank.nsmallest(3, "ranking")[["ranking", "AI", "score_final"]]
print("Top 3 systems by composite score:")
for _, r in top3.iterrows():
    print(f"#{int(r['ranking'])} {r['AI']} ? score_final={r['score_final']:.3f}")

### Portfolio interpretation

- The ranking is **relative within this benchmark design**, not a universal leaderboard.
- High-performing systems are those that best balance **readability + semantic similarity + lexical alignment** with OMS content.


## 4. Topic-level performance

Global ranking hides topic specialization. The table below shows the best system per topic using `score_topic`.


In [ ]:
best_by_topic = (
    df_agg_topic.sort_values(["topic", "score_topic"], ascending=[True, False])
    .groupby("topic", as_index=False)
    .first()[["topic", "AI", "score_topic"]]
    .rename(columns={"AI": "best_AI"})
)
display(best_by_topic)

topic_spread = (
    df_agg_topic.groupby("topic")["score_topic"]
    .agg(["min", "max", "mean", "std"])
    .reset_index()
)
display(topic_spread)

## 5. Cluster analysis and statistical evidence

The original project selected `k=3` clusters (Elbow + Silhouette) and reported strong group differences.

Below we reproduce a compact statistical check from the exported topic-cluster table.


In [ ]:
cluster_summary = (
    df_agg_topic_cluster.groupby("Cluster")["score_topic"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
    .sort_values("mean")
)
display(cluster_summary)

# One-way ANOVA on topic scores across clusters
groups = [
    df_agg_topic_cluster.loc[df_agg_topic_cluster["Cluster"] == c, "score_topic"].values
    for c in sorted(df_agg_topic_cluster["Cluster"].unique())
]
F_stat, p_value = f_oneway(*groups)
print(f"ANOVA on score_topic by Cluster: F={F_stat:.4f}, p-value={p_value:.6g}")

## 6. Visual highlights

Representative charts generated by the original project script:

- Final ranking
- Ranking by topic
- Elbow and Silhouette methods
- 2D MCA perceptual map


In [ ]:
from matplotlib.image import imread

img_files = [
    "08_ai_final_ranking.png",
    "10_ai_ranking_by_topic.png",
    "11_elbow_method.png",
    "12_silhouette_method.png",
    "15_mca_perceptual_map_2d.png",
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for ax, name in zip(axes, img_files):
    p = FIGURES / name
    if p.exists():
        ax.imshow(imread(p))
        ax.set_title(name)
        ax.axis("off")
    else:
        ax.text(0.5, 0.5, f"Missing: {name}", ha="center", va="center")
        ax.axis("off")

# Hide any remaining empty subplot
for j in range(len(img_files), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

## 7. Final takeaways

- The methodology was effective in distinguishing **stable vs. unstable** model behavior in a sensitive health-information context.
- Results support that **topic formulation** can strongly affect response quality patterns.
- Combining complementary metrics + clustering + inferential tests is a robust framework for auditing generative AI outputs.

## 8. Reproducibility and full artifacts

- Full implementation: `../code/TCC_MBA_DSA_USP_MALARIA.py`
- Raw/processed data: `../data/` and `../results/`
- Figures: `../figures/`

This notebook is intentionally concise and intended for portfolio communication.
